## BASE

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
import time
from tqdm.notebook import tqdm
from collections import defaultdict

In [ ]:
DATA_ROOT = "/home/alex/internship/datasets/aqua20/data/aqua20"
NUM_CLASSES = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 252

In [ ]:
transform = transforms.Compose([
    transforms.Resize(RESOLUTION),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False
backbone = backbone.to(DEVICE)

Using cache found in /home/alex/.cache/torch/hub/facebookresearch_dinov2_main
/home/alex/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/alex/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/alex/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [ ]:
def extract_features(loader, desc="Extracting features"):
    all_feats, all_labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=desc, leave=False):
            all_feats.append(backbone(x.to(DEVICE)).cpu())
            all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)


def train_linear_probe(train_feats, train_labels, test_feats, test_labels,
                       epochs=50, lr=1e-3, eval_every=10):
    head = nn.Linear(768, NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Loaders sur features précalculées — tout en RAM, très rapide
    train_feat_loader = DataLoader(
        TensorDataset(train_feats, train_labels), batch_size=256, shuffle=True
    )
    test_feat_loader = DataLoader(
        TensorDataset(test_feats, test_labels), batch_size=256, shuffle=False
    )

    history = defaultdict(list)

    for epoch in tqdm(range(epochs), desc="Training"):
        head.train()
        total_loss, correct, total = 0.0, 0, 0

        for feats, y in train_feat_loader:
            feats, y = feats.to(DEVICE), y.to(DEVICE)
            logits = head(feats)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += len(y)

        if (epoch + 1) % eval_every == 0 or epoch == epochs - 1:
            head.eval()
            all_preds, all_labels_val = [], []
            with torch.no_grad():
                for feats, y in test_feat_loader:
                    feats = feats.to(DEVICE)
                    preds = head(feats).argmax(dim=1).cpu()
                    all_preds.append(preds)
                    all_labels_val.append(y)

            all_preds      = torch.cat(all_preds).numpy()
            all_labels_val = torch.cat(all_labels_val).numpy()

            f1_macro = f1_score(all_labels_val, all_preds, average="macro")
            f1_weighted = f1_score(all_labels_val, all_preds, average="weighted")

            tqdm.write(
                f"Epoch {epoch+1:>3}/{epochs} | "
                f"Loss: {total_loss/total:.4f} | "
                f"Train Acc: {correct/total*100:.1f}% | "
                f"F1 Macro: {f1_macro*100:.1f}% | "
                f"F1 Weighted: {f1_weighted*100:.1f}%"
            )
            history["epoch"][epoch + 1] = {
                "loss": total_loss / total,
                "train_acc": correct / total,
                "f1_macro": f1_macro,
                "f1_weighted": f1_weighted
            }

    return head, history

In [ ]:
test_ds    = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=transform)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
test_feats,    test_labels    = extract_features(test_loader,    "Test")



## Baselines

### Full Data

In [ ]:
full_train = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=transform)
full_loader = DataLoader(full_train, batch_size=64, shuffle=True)

In [ ]:
print("\nTraining on full data...")
start_time = time.time()
full_feats,    full_labels    = extract_features(full_loader,    "Full train")
head_full, history_full = train_linear_probe(full_feats, full_labels, test_feats, test_labels,
                                epochs=50, eval_every=10)
full_total_time = time.time() - start_time
print(f"Training time: {full_total_time:.6f} seconds")

## Setup multiple


In [ ]:
DISTILLED_BASE_DIR = "../logged_files/distillation/aqua20/dinov2_vitb"

In [ ]:
class DistilledRun():

    def __init__(self, distilled_run: str, batch_size: int = 20):
        self.distilled_run: str = distilled_run
        self.syn_data: dict[str, torch.Tensor] = torch.load(
            f"{DISTILLED_BASE_DIR}/{distilled_run}/data.pth", 
            map_location="cpu"
        )
        self.distill_loader: DataLoader = DataLoader(
            TensorDataset(self.syn_data["images"], self.syn_data["labels"]),
            batch_size=batch_size, shuffle=False
        )
        self.total_time: float = 0.0
        self.head: nn.Linear | None = None
        self.history: dict | None = None

    def run_training(self, test_feats, test_labels, epochs=50, lr=1e-3, eval_every=10):
        start_time = time.time()
        feats, labels = extract_features(self.distill_loader, desc=f"Extracting features for {self.distilled_run}")
        self.head, self.history = train_linear_probe(feats, labels, test_feats, test_labels,
                                  epochs=epochs, lr=lr, eval_every=eval_every)
        self.total_time = time.time() - start_time
    
    def to_dict(self):
        return {"total_time": self.total_time, "history": self.history}
    
    def __str__(self):
        return f"DistilledRun(run={self.distilled_run}, total_time={self.total_time:.2f}s)"

In [ ]:
import re
from pathlib import Path

distilled_run_regex = re.compile(r"^distill_aqua20_h100_\d+ipc(?:_(?:physics|seathru))?$")

distilled_runs: list[DistilledRun] = [
    DistilledRun(p.name)
    for p in sorted(Path(DISTILLED_BASE_DIR).iterdir())
    if p.is_dir() and distilled_run_regex.match(p.name)
]

print(f"Found {len(distilled_runs)} distilled runs:")

In [ ]:
for run in distilled_runs:
    print(f"Running training for {run.distilled_run}...")
    run.run_training(epochs=50, lr=1e-3, eval_every=10)
    print(f"Finished {run.distilled_run} in {run.total_time:.2f}s\n")

In [ ]:
history = {run.distilled_run: run.to_dict() for run in distilled_runs}

history["full_data"] = {"total_time": full_total_time, "history": history_full}